# 3. Data Preparation: Preprocessing, Augmentasi & Pembagian Dataset

Tahap **Data Preparation** mempersiapkan tensor citra agar siap dilatih oleh arsitektur Convolutional Neural Network (CNN):
1. **Verifikasi Integritas & Pembersihan Data:** Memvalidasi citra corrupt/rusak dan menghapus file tidak valid.
2. **Penyelarasan Dimensi (*Image Resizing*):** Mengubah seluruh citra ke resolusi standar input CNN ($224 	imes 224$ piksel).
3. **Normalisasi Nilai Piksel:** Melakukan scaling intensitas piksel ke rentang $[0, 1]$ atau standardisasi ImageNet ($\mu=[0.485, 0.456, 0.406]$, $\sigma=[0.229, 0.224, 0.225]$).
4. **Teknik Augmentasi Data:** Menerapkan transformasi acak (*Random Rotation, Random Horizontal Flip, Color Jitter, Random Zoom*) untuk meningkatkan ketahanan model.
5. **Stratifikasi Dataset (*Train-Val-Test Split*):** Membagi data dengan rasio $70\% : 15\% : 15\%$ secara berimbang per kelas.


In [ ]:
import os
import shutil
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from PIL import Image

RAW_DIR = Path('data/raw')
PROCESSED_DIR = Path('data/processed')
TARGET_SIZE = (224, 224)
SEED = 42


---
## 3.1 Skema Pembagian Dataset (Stratified Train, Validation, Test Split)


In [ ]:
def prepare_splits(raw_dir, processed_dir, split_ratios=(0.70, 0.15, 0.15)):
    if not raw_dir.exists() or not any(raw_dir.iterdir()):
        print(f"Folder {raw_dir} masih kosong. Masukkan dataset yang diunduh ke dalam folder tersebut.")
        return
        
    records = []
    valid_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
    for class_folder in sorted([d for d in raw_dir.iterdir() if d.is_dir()]):
        for img_path in class_folder.glob('*.*'):
            if img_path.suffix.lower() in valid_exts:
                records.append({
                    'filepath': str(img_path),
                    'class_name': class_folder.name
                })
    df = pd.DataFrame(records)
    if df.empty:
        print("Tidak ada citra valid ditemukan.")
        return
        
    # Split Train vs (Val + Test)
    train_df, temp_df = train_test_split(
        df, test_size=(split_ratios[1] + split_ratios[2]),
        stratify=df['class_name'], random_state=SEED
    )
    
    # Split Val vs Test
    val_ratio_in_temp = split_ratios[1] / (split_ratios[1] + split_ratios[2])
    val_df, test_df = train_test_split(
        temp_df, test_size=(1.0 - val_ratio_in_temp),
        stratify=temp_df['class_name'], random_state=SEED
    )
    
    print(f"Total data: {len(df)}")
    print(f"Train set : {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)")
    print(f"Val set   : {len(val_df)} ({len(val_df)/len(df)*100:.1f}%)")
    print(f"Test set  : {len(test_df)} ({len(test_df)/len(df)*100:.1f}%)")
    
    # Simpan manifest metadata CSV
    train_df['split'] = 'train'
    val_df['split'] = 'val'
    test_df['split'] = 'test'
    manifest = pd.concat([train_df, val_df, test_df])
    manifest.to_csv(processed_dir / 'dataset_manifest.csv', index=False)
    print(f"Manifest tersimpan di: {processed_dir / 'dataset_manifest.csv'}")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
prepare_splits(RAW_DIR, PROCESSED_DIR)


---
## 3.2 Pembuatan Data Pipeline & Augmentasi (PyTorch / TensorFlow / Keras Ready)


In [ ]:
print("Kerangka Data Pipeline & Augmentasi siap dikonfigurasi saat data telah diunduh.")
